# Conexión Mongo DB

In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['proyecto_bigdata'] 
coleccion = db['AutoTec_scraping'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


# Conectar el Scraper a la BD y guardar
Se usa el script de la semana 3

In [15]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time #para capturar la hora y fecha de obtención

NOMBRE_GRUPO = "AutoTec" #Agegaremos el nombre de grupo dentro del diccionario y la hora en que se obtubo
CATEGORIA_ACTUAL = "Automoviles" # Los alum
# --- PASO 1: CONFIGURACIÓN ---
options = Options()
options.add_argument("--headless") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

# --- PASO 2: INICIALIZACIÓN DE VARIABLES (LA MEMORIA) ---
lista_autos = []
limite_paginas = 3

try:
    driver.get("https://www.yapo.cl/autos-usados")
    
    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Nivel de Profundidad {nivel_pagina + 1} ---")
        
        # 1. Espera al Contenedor Raíz (el que agrupa todos los resultados)
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "d3-ad-tile__content"))
        )

        # 2. Captura de Bloques de Primer Nivel (la "tarjeta" o "card" del item)
        bloques_primer_nivel = driver.find_elements(By.CLASS_NAME, "d3-ad-tile__content")
        
        for bloque in bloques_primer_nivel:
            # 3. Búsqueda Detallada dentro del bloque (Selectores específicos)
            # Extraemos el atributo 'title' del enlace para evitar texto truncado
            elemento_titulo = bloque.find_element(By.CLASS_NAME, "d3-ad-tile__title")
            dato_identificador = elemento_titulo.text.strip()
            
            dato_valor = bloque.find_element(By.CLASS_NAME, "d3-ad-tile__price").text
            
            # Guardamos la estructura en nuestra memoria
            item_extraido = {
                "identificador": dato_identificador,
                "valor": dato_valor,
                "categoria": CATEGORIA_ACTUAL,
                "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                "grupo": NOMBRE_GRUPO
            }
            lista_autos.append(item_extraido)

        # --- NAVEGACIÓN: Lógica de Salto de Página ---
        try:
            print("Buscando botón Siguiente...")
            
            # 1. Bajamos hasta el final de la página para que el botón se cargue/renderice
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2) 

            # 2. Intentamos encontrar el botón por varios selectores comunes en Yapo
            # El selector "svg" y el aria-label son los más consistentes actualmente
            disparador_siguiente = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "button[aria-label='Next page'], a[aria-label='Next page'], .pagination-next, button[class*='next']"))
            )

            # 3. Forzamos el click mediante JavaScript (ignora si hay algo encima tapándolo)
            driver.execute_script("arguments[0].click();", disparador_siguiente)
            
            # 4. Espera crucial para que la nueva lista de autos cargue antes de la siguiente iteración
            time.sleep(4)
            print("Salto de página exitoso.")
            
        except Exception as e:
            print(f"No se encontró más paginación o el botón cambió. Deteniendo scraping.")
            break 

    print(f"\nExtracción finalizada. Registros en memoria: {len(lista_autos)}")

except Exception as e:
    print(f"Fallo en la arquitectura de scraping: {e}")

finally:
    driver.quit()

#TENEMOS EL DICCIONARIO CON UN FORMATO JSON, OBTENIDO DEL SCRIPT. AHORA LO INCLUIMOS EN LA BASE DE DATOS
#Cambios en los datos de salida
from pymongo import MongoClient



# 1. Configuracion de conexion
try:
    # 'database' es el nombre del servicio en el docker-compose
    client = MongoClient('database', 27017)
    db = client['proyecto_bigdata']
    coleccion = db['AutoTec_scraping']
    print("CONEXION ESTABLECIDA: Listo para insertar datos de",CATEGORIA_ACTUAL)
except Exception as e:
    print("ERROR DE CONEXION:", e)

# 2. Procesamiento e insercion de datos
registros_exitosos = 0
registros_fallidos = 0

for dato in lista_autos:
    try:
        # Limpieza de simbolos y conversion a float
        # Se eliminan simbolos de moneda, comas y espacios en blanco
        valor_sucio = str(dato.get('valor', '0'))
        valor_limpio = valor_sucio.split('\n')[0] # Toma solo la primera línea si hay descuento
        valor_limpio = valor_limpio.replace('$', '').replace('.', '').replace(',', '').strip()        
        dato['valor'] = float(valor_limpio)
        
        # Insercion individual
        coleccion.insert_one(dato)
        registros_exitosos += 1
        
    except ValueError:
        print("ADVERTENCIA: No se pudo procesar el valor:", valor_sucio)
        registros_fallidos += 1
    except Exception as e:
        print("ERROR EN INSERCION:", e)
        registros_fallidos += 1

print("RESUMEN DE CARGA:")
print("Registros exitosos:", registros_exitosos)
print("Registros fallidos:", registros_fallidos)

--- Procesando Nivel de Profundidad 1 ---
Buscando botón Siguiente...
Salto de página exitoso.
--- Procesando Nivel de Profundidad 2 ---
Buscando botón Siguiente...
Salto de página exitoso.
--- Procesando Nivel de Profundidad 3 ---
Buscando botón Siguiente...
Salto de página exitoso.

Extracción finalizada. Registros en memoria: 90
CONEXION ESTABLECIDA: Listo para insertar datos de Automoviles
RESUMEN DE CARGA:
Registros exitosos: 90
Registros fallidos: 0
